In [622]:
import pandas as pd
from pathlib import Path

folder = Path("../extract/arquivos_brutos/")

df_pts = pd.read_parquet(folder / 'df_partidas.parquet')
df_pts.head(4)

,0,1,2,3,4,5,6,7,8,9,10,11
0,,CAMPEONATO,DATA,TIER,STAGE,GAME TIME,TIME A,TIME B,FORMATO (BO:1/3/5),RESULTADO,PTS TIME A,PTS TIME B
1,,,,,,,,,,,,
2,,Campeonato Mineiro de Esportes Eletrônicos 2026,05/07/2026,TIER S,KNOCKOUT STAGE,,-,-,BO5,A,20,-20
3,,Campeonato Mineiro de Esportes Eletrônicos 2026,05/07/2026,TIER S,KNOCKOUT STAGE,,-,-,BO5,A,0,0


In [623]:
df_pts.columns

Index([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], dtype='int64')

In [624]:
#refazendo colunas
df_pts.columns = df_pts.iloc[0]
df_pts = df_pts.iloc[2:].reset_index(drop=True)


In [625]:
#tratando colunas
df_pts.columns = [
  valor.lower().replace(" ", "_").replace("/", "-").replace("(", "").replace(")", "")
  for valor in df_pts.columns
]


In [626]:
#Removendo os "-" de colunas de times vazios:

colunas = ["time_a", "time_b"]
df_pts[colunas] = df_pts[colunas].replace("-", pd.NA)


In [627]:
#Removendo linhas que possuem os 2 times vazios
df_pts[colunas] = df_pts[colunas].apply(
  lambda coluna: coluna.str.strip()
).replace(["", "-"],pd.NA)

In [628]:
df_pts = df_pts.dropna(
    subset=["time_a", "time_b"],
    how="all"
)

In [629]:
#Inserindo coluna de id para partidas
df_pts.insert(loc=0, column='id_partida', value=0)


In [630]:
#Adicionando valor a coluna de id
df_pts["id_partida"] = (
 range(1, len(df_pts) + 1)
)

In [631]:
#Convertendo coluna de data para datetime
df_pts['data'] = pd.to_datetime(
  df_pts['data'].str.strip(),
  format= "mixed",
  dayfirst=True,
  errors="raise"
)

In [632]:
# Trata valores vazios
tempos = df_pts['game_time'].str.strip().replace(
    ['', '-'], pd.NA
)

# Separa minutos, segundos e centésimos
partes = tempos.str.split(':', expand=True).astype('Float64')

# Converte diretamente para minutos decimais
df_pts['game_time'] = (
    partes[0]
    + partes[1] / 60
    + partes[2] / 6000
)

In [633]:
#Renomeando coluna de formato de jogo
df_pts = df_pts.rename(
  columns= {"formato_bo:1-3-5" : "match_format"}
)

In [634]:
#Convertendo as colunas de pontuaçao para int
colunas =['pts_time_a', 'pts_time_b']

for coluna in colunas:
  df_pts[coluna] = pd.to_numeric(
    df_pts[coluna], errors="coerce"
  ).astype("Int64")

In [635]:
#Sequencia de true e false para os times que ganharam ou nao
venceu_a = df_pts['resultado'] == 'A'
venceu_b = df_pts['resultado'] == 'B'

In [636]:
#Criando a condicao para substituir as letras pelos times:
#Nas linhas em que A venceu, preencha resultado com o valor de time_a da mesma linha.
df_pts.loc[venceu_a, 'resultado'] = df_pts.loc[venceu_a, 'time_a']

#Nas linhas em que B venceu, preencha resultado com o valor de time_b da mesma linha.
df_pts.loc[venceu_b, 'resultado'] = df_pts.loc[venceu_b, 'time_b']

In [637]:
df_pts.head(4)

,id_partida,,campeonato,data,tier,stage,game_time,time_a,time_b,match_format,resultado,pts_time_a,pts_time_b
3,1,,Campeonato Mineiro de Esportes Eletrônicos 2026,2026-07-05,TIER S,KNOCKOUT STAGE,<NA>,MIG,GHOST GAMING,BO3,MIG,15,-15
4,2,,Campeonato Mineiro de Esportes Eletrônicos 2026,2026-07-05,TIER S,KNOCKOUT STAGE,<NA>,MIG,GHOST GAMING,BO3,MIG,0,0
5,3,,Campeonato Mineiro de Esportes Eletrônicos 2026,2026-07-05,TIER S,KNOCKOUT STAGE,<NA>,FEAR CLAN,GEN MG,BO3,GEN MG,-15,15
6,4,,Campeonato Mineiro de Esportes Eletrônicos 2026,2026-07-05,TIER S,KNOCKOUT STAGE,<NA>,FEAR CLAN,GEN MG,BO3,GEN MG,0,0


In [638]:
import re
import unicodedata

def normalizar_coluna(coluna):
  #Converter a coluna para string, remover espaços no inicio e no final, converter para minúsculas
  coluna = str(coluna).strip().lower()

  #Decomposição, retira acentos da palavra, ã vira a + ~
  coluna = unicodedata.normalize("NFKD", coluna)
  
  #Remove os acentos
  coluna = "".join(
    c for c in coluna
    if not unicodedata.combining(c)
  )

  coluna = re.sub(r"[^a-z0-9]+", "_", coluna)
  return coluna.strip("_")


In [639]:
#Ao inves de passar a coluna inteira para o metodo, aplicar o metodo a cada valor da coluna
df_pts['campeonato'] = df_pts['campeonato'].apply(normalizar_coluna)
df_pts['tier'] = df_pts['tier'].apply(normalizar_coluna)
df_pts['stage'] = df_pts['stage'].apply(normalizar_coluna)
df_pts['time_a'] = df_pts['time_a'].apply(normalizar_coluna)
df_pts['time_b'] = df_pts['time_b'].apply(normalizar_coluna)
df_pts['resultado'] = df_pts['resultado'].apply(normalizar_coluna)



In [640]:
equivalencias_campeonatos = {
  "sesc_geek": "sesc_geek_2024",
  
  "cmee_2025_conferencia_metropolitana":
        "campeonato_mineiro_de_esportes_eletronicos_2025_conferencia_metropolitana",

  "cmee_2025_conferencia_lcq":
    "campeonato_mineiro_de_esportes_eletronicos_2025_last_chance_qualify",

  "cmee_2025_conferencia_triangulo_mineiro":
    "campeonato_mineiro_de_esportes_eletronicos_2025_conferencia_triangulo",

  "liga_dinoco_s": "liga_dinoco_s_season_i",

  "desafio_uni_lol_circuito_academico_uni_bh":
    "uni_lol_circuito_academico_uni_bh",

  "arena_goat_lol": "arena_goat_lol_season_ii"
}

df_pts['campeonato'] = df_pts['campeonato'].replace(
  equivalencias_campeonatos
)

In [641]:
#Dimensao de torneios para fazer o cruzamento de dados
dim_torneios = pd.read_excel("excel_transform/campeonatos/Campeonatos.xlsx")


In [642]:
dim_torneios[['id_camp', 'nome_do_campeonato']].head()

,id_camp,nome_do_campeonato
0,0,campeonato_mineiro_de_esportes_eletronicos_2026
1,1,campeonato_mineiro_de_esportes_eletronicos_202...
2,2,campeonato_mineiro_de_esportes_eletronicos_202...
3,3,campeonato_mineiro_de_esportes_eletronicos_2025
4,4,campeonato_mineiro_de_esportes_eletronicos_202...


In [643]:
#Identificar valores repetidos e devolver uma sequencia de True e False
duplicados = dim_torneios['nome_do_campeonato'].duplicated(keep=False)

#Mostrar apenas as linhas marcadas como duplicadas, nas colunas que vou analisar
dim_torneios.loc[duplicados,['id_camp', 'nome_do_campeonato']]

,id_camp,nome_do_campeonato


In [644]:
#Realizando o cruzamento: Para cada partida, procurar o campeonato e trazer seu id:
partidas_com_torneio = df_pts.merge(
  dim_torneios[['id_camp', 'nome_do_campeonato']], #Seleciona ID e nome do campeonato
  left_on='campeonato',                            #Indica a coluna a esquerda a comparar
  right_on='nome_do_campeonato',                   #Indica a coluna a direita a comparar
  how='left',                                      #Mantem todas as partidas, se nao encontrar o camp, o id fica ausente
  validate='many_to_one',                          #Permite varias partidas para um mesmo torneio
  indicator=True                                   #Indica se houve correspondencia
)

In [645]:
partidas_com_torneio['_merge'].value_counts()

_merge
both          525
left_only       0
right_only      0
Name: count, dtype: int64

In [646]:
#Transformando as informaçoes de campeonatos em IDs
df_pts = (
  partidas_com_torneio
  .rename(columns={"id_camp": "tournament_id"})
  .drop(columns=['campeonato', 'nome_do_campeonato', '_merge'])
  .copy()
)

df_pts['tournament_id'] = df_pts['tournament_id'].astype('int64')

In [647]:
df_pts[['id_partida', 'tournament_id', 'data']].head(6)

,id_partida,tournament_id,data
0,1,0,2026-07-05
1,2,0,2026-07-05
2,3,0,2026-07-05
3,4,0,2026-07-05
4,5,0,2026-07-04
5,6,0,2026-07-04


In [648]:
df_pts

,id_partida,,data,tier,stage,game_time,time_a,time_b,match_format,resultado,pts_time_a,pts_time_b,tournament_id
0,1,,2026-07-05,tier_s,knockout_stage,<NA>,mig,ghost_gaming,BO3,mig,15,-15,0
1,2,,2026-07-05,tier_s,knockout_stage,<NA>,mig,ghost_gaming,BO3,mig,0,0,0
2,3,,2026-07-05,tier_s,knockout_stage,<NA>,fear_clan,gen_mg,BO3,gen_mg,-15,15,0
3,4,,2026-07-05,tier_s,knockout_stage,<NA>,fear_clan,gen_mg,BO3,gen_mg,0,0,0
4,5,,2026-07-04,tier_s,group_stage,<NA>,mig,verdonha,BO1,mig,10,-10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
520,521,,2017-01-20,tier_c,fase_2,<NA>,autofeed,team_bronze,BO1,team_bronze,-4,4,33
521,522,,2017-01-20,tier_c,fase_2,<NA>,adcepcao,lishosac,BO1,lishosac,-4,4,33
522,523,,2017-01-20,tier_c,fase_2,<NA>,sinners,legends_team,BO1,legends_team,-4,4,33
523,524,,2017-01-20,tier_c,fase_1,<NA>,skyteam,suit_up,BO1,suit_up,-4,4,33


In [649]:
dim_times = pd.read_excel('../transform/excel_transform/times/Times.xlsx')
dim_times.head(3)

,time_id,nome_da_equipe,ranking_atual_automatico,ranking_base,penalidade_inatividade
0,1,fear_clan,466.03,100.0,0.71
1,2,mig,445.50,100.0,0.10
2,3,ghost_gaming,425.83,100.0,0.03


In [650]:
#Como encontramos times com seus nomes nao normalizados, estamos fazendo isso agr
dim_times['nome_da_equipe'] = (
  dim_times['nome_da_equipe'].apply(normalizar_coluna)
)

In [651]:
#Conferindo se existe pelo menos alguma linha do df que tenha apenas 1 time preenchido
filtro = df_pts['time_a'].notna() ^ df_pts['time_b'].notna()

df_filtr = df_pts[filtro]
df_filtr

,id_partida,,data,tier,stage,game_time,time_a,time_b,match_format,resultado,pts_time_a,pts_time_b,tournament_id


In [652]:
#Verificando nomes duplicados na dimensao de times:
filtro_times_duplicados = dim_times['nome_da_equipe'].duplicated(keep=False)

#Exibe os nomes duplicados
dim_times.loc[filtro_times_duplicados, ['time_id', 'nome_da_equipe']]

,time_id,nome_da_equipe


In [653]:
equivalencias_equipes = {
    'fumec_e_sports': 'fumec_esports',
    'lishosac': 'lishos_ac',
    'indominus_team':'team_indominus',
    'mig_blackburn': 'mig',
    'squad_pao_de_queijo' : 'pao_de_queijo_e_sports',
    'dino_monster' : 'team_monster',
    'team_dignos' : 'team_dig'
}

df_pts['resultado'] = df_pts['resultado'].replace(equivalencias_equipes)
df_pts['time_a'] = df_pts['time_a'].replace(equivalencias_equipes)
df_pts['time_b'] = df_pts['time_b'].replace(equivalencias_equipes)

In [654]:
#Selecionando as colunas necessárias e renomeia o ID pro time a
equipes_a = dim_times[['time_id', 'nome_da_equipe']].rename(
  columns={'time_id': 'team_a_id'}
)



In [655]:
#Cruzando agora as partidas do df de partidas, com o filtro equipes_a feito acima:
partidas_com_time_a = df_pts.merge(
  equipes_a,
  left_on='time_a', #Campo do dataframe de partidas
  right_on='nome_da_equipe', #Campo do dataframe de times
  how='left', #Mantem todas as partidas (LEFT JOIN)
  validate='many_to_one', #varias partidas (esquerda) correspondam a mesma equipe (direita), mas nomes 1 vez apenas em equipe_a.
  indicator=True #Criando a coluna _merge para avaliar se deu certo

)

In [656]:
df_pts = partidas_com_time_a.drop(
  columns=['nome_da_equipe', '_merge']
).copy()

df_pts['team_a_id'] = df_pts['team_a_id'].astype('Int64')

In [657]:
#Selecionando as colunas necessárias e renomeia o ID pro time b
equipes_b = dim_times[['time_id', 'nome_da_equipe']].rename(
  columns={'time_id': 'team_b_id'}
)

In [658]:
#Cruzando agora as partidas do df de partidas, com o filtro equipes_a feito acima:
partidas_com_time_b = df_pts.merge(
  equipes_b,
  left_on='time_b', #Campo do dataframe de partidas
  right_on='nome_da_equipe', #Campo do dataframe de times
  how='left', #Mantem todas as partidas (LEFT JOIN)
  validate='many_to_one', #varias partidas (esquerda) correspondam a mesma equipe (direita), mas nomes 1 vez apenas em equipe_a.
  indicator=True #Criando a coluna _merge para avaliar se deu certo
)

In [659]:
partidas_com_time_b['_merge'].value_counts()

_merge
both          525
left_only       0
right_only      0
Name: count, dtype: int64

In [660]:
partidas_com_time_b.loc[
    partidas_com_time_b['_merge'] == 'left_only',
    'time_b'
].value_counts(dropna=False)

Series([], Name: count, dtype: int64)

In [661]:
#Dropando o nome do time b pelo seu id vindo de dim_times
df_pts = partidas_com_time_b.drop(
  columns=['nome_da_equipe', '_merge']
).copy()
df_pts['team_b_id'] = df_pts['team_b_id'].astype('Int64')

In [662]:
#Avaliando como ficou a df_pts filtrando pelos campos novos
df_pts[
    ['id_partida', 'time_a', 'team_a_id', 'time_b', 'team_b_id']
].head(10)

,id_partida,time_a,team_a_id,time_b,team_b_id
0,1,mig,2,ghost_gaming,3
1,2,mig,2,ghost_gaming,3
2,3,fear_clan,1,gen_mg,4
3,4,fear_clan,1,gen_mg,4
4,5,mig,2,verdonha,5
5,6,oab_para_de_minas,7,gen_mg,4
6,7,inatel_e_sports,8,fear_clan,1
7,8,unifal_cientec,6,ghost_gaming,3
8,9,verdonha,5,gen_mg,4
9,10,oab_para_de_minas,7,mig,2


In [663]:
#Avaliando quais times ficaram sem correspondencia
df_pts[['team_a_id', 'team_b_id']].isna().sum()

team_a_id    0
team_b_id    0
dtype: int64

In [664]:
#Avaliando como ficou a df_pts filtrando pelos campos novos
df_pts[
    ['id_partida', 'time_a', 'team_a_id', 'time_b', 'team_b_id', 'resultado']
].head(10)

,id_partida,time_a,team_a_id,time_b,team_b_id,resultado
0,1,mig,2,ghost_gaming,3,mig
1,2,mig,2,ghost_gaming,3,mig
2,3,fear_clan,1,gen_mg,4,gen_mg
3,4,fear_clan,1,gen_mg,4,gen_mg
4,5,mig,2,verdonha,5,mig
5,6,oab_para_de_minas,7,gen_mg,4,gen_mg
6,7,inatel_e_sports,8,fear_clan,1,fear_clan
7,8,unifal_cientec,6,ghost_gaming,3,ghost_gaming
8,9,verdonha,5,gen_mg,4,gen_mg
9,10,oab_para_de_minas,7,mig,2,mig


In [665]:
#Comparar as 2 colunas para ver quano o time A venceu
venceu_time_a = df_pts['resultado'].eq(df_pts['time_a'])


In [666]:
venceu_time_b = df_pts['resultado'].eq(df_pts['time_b'])

In [667]:
df_pts.loc[
    venceu_time_a,
    ['time_a', 'team_a_id', 'resultado']
].head()

,time_a,team_a_id,resultado
0,mig,2,mig
1,mig,2,mig
4,mig,2,mig
10,inatel_e_sports,8,inatel_e_sports
14,fear_clan,1,fear_clan


In [668]:
df_pts.loc[
  venceu_time_b,
  ['time_b', 'team_b_id', 'resultado']
].head()

,time_b,team_b_id,resultado
2,gen_mg,4,gen_mg
3,gen_mg,4,gen_mg
5,gen_mg,4,gen_mg
6,fear_clan,1,fear_clan
7,ghost_gaming,3,ghost_gaming


In [669]:
#Criar a coluna de ids inteiros, vazia
df_pts['winner_team_id'] = pd.Series(
  pd.NA,
  index=df_pts.index,
  dtype='Int64'
)

In [670]:
#Nas partidas em que o A venceu, copia o id do time A
df_pts.loc[venceu_time_a, 'winner_team_id'] = (
  df_pts.loc[venceu_time_a, 'team_a_id']
)

In [671]:
#Nas partidas em que o B venceu, copia o id do time B
df_pts.loc[venceu_time_b, 'winner_team_id'] = (
  df_pts.loc[venceu_time_b, 'team_b_id']
)

In [672]:
df_pts[
    ['time_a', 'team_a_id', 'time_b', 'team_b_id', 'resultado', 'winner_team_id']
].head(10)

,time_a,team_a_id,time_b,team_b_id,resultado,winner_team_id
0,mig,2,ghost_gaming,3,mig,2
1,mig,2,ghost_gaming,3,mig,2
2,fear_clan,1,gen_mg,4,gen_mg,4
3,fear_clan,1,gen_mg,4,gen_mg,4
4,mig,2,verdonha,5,mig,2
5,oab_para_de_minas,7,gen_mg,4,gen_mg,4
6,inatel_e_sports,8,fear_clan,1,fear_clan,1
7,unifal_cientec,6,ghost_gaming,3,ghost_gaming,3
8,verdonha,5,gen_mg,4,gen_mg,4
9,oab_para_de_minas,7,mig,2,mig,2


In [673]:
sem_vencedor_identificado = ~(venceu_time_a | venceu_time_b)

df_pts.loc[
  sem_vencedor_identificado,
  ['id_partida', 'time_a', 'time_b', 'resultado']
]

,id_partida,time_a,time_b,resultado


In [674]:
#Verificar quais vencedores ficaram sem ID:
df_pts.loc[
  df_pts['winner_team_id'].isna(),
  ['id_partida', 'time_a', 'time_b', 'resultado']
]

,id_partida,time_a,time_b,resultado


In [675]:
pendencias_equipes = df_pts.loc[
  df_pts[['team_a_id', 'team_b_id', 'winner_team_id']]
  .isna()
  .any(axis=1)
].copy()

In [676]:
#Salvando equipes pendentes para tratá-las posteriormente:
pendencias_equipes.to_csv(
  'pendencias_equipes.csv',
  index=False,
  encoding='utf-8-sig'
)

In [677]:
nomes_pendentes = pd.concat([
    pendencias_equipes.loc[
        pendencias_equipes['team_a_id'].isna(), 'time_a'
    ],
    pendencias_equipes.loc[
        pendencias_equipes['team_b_id'].isna(), 'time_b'
    ]
]).drop_duplicates().sort_values()

nomes_pendentes

Series([], dtype: str)

In [678]:
df_pts.loc[
    df_pts['team_a_id'].isna() | df_pts['team_b_id'].isna(),
    ['id_partida', 'data', 'tournament_id', 'time_a', 'time_b', 'resultado']
].sort_values('data')

,id_partida,data,tournament_id,time_a,time_b,resultado


In [679]:
pontuacao_bruta = pd.read_parquet(
    '../extract/arquivos_brutos/df_pontuacao.parquet'
)

# Busca em todas as colunas, mantendo o conteúdo original
filtro = pontuacao_bruta.astype('string').apply(
    lambda coluna: coluna.str.contains(
        r'blackburn|monster|queijo|dignos|team.?dig',
        case=False,
        na=False,
        regex=True
    )
).any(axis=1)

pontuacao_bruta.loc[filtro]

,0,1,2,3,4,5,6,7,8,9,...,49,50,51,52,53,54,55,56,57,58
18,TEAM MONSTER,"50,4","55,00%",100,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
46,BLACKBURN,"57,15","55,00%",100,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
158,PÃO DE QUEIJO E-SPORTS,36,"55,00%",100,-,-20,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
195,TEAM DIG,"45,9","55,00%",100,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-


In [680]:
df_pts = df_pts.drop(
    columns=['time_a', 'time_b', 'resultado']
)
    

In [681]:
df_pts[
    ['id_partida', 'team_a_id', 'team_b_id', 'winner_team_id']
].head()

,id_partida,team_a_id,team_b_id,winner_team_id
0,1,2,3,2
1,2,2,3,2
2,3,1,4,4
3,4,1,4,4
4,5,2,5,2


In [682]:
df_pts = df_pts.rename(columns={'id_partida': 'match_id' })

In [683]:
df_pts

,match_id,,data,tier,stage,game_time,match_format,pts_time_a,pts_time_b,tournament_id,team_a_id,team_b_id,winner_team_id
0,1,,2026-07-05,tier_s,knockout_stage,<NA>,BO3,15,-15,0,2,3,2
1,2,,2026-07-05,tier_s,knockout_stage,<NA>,BO3,0,0,0,2,3,2
2,3,,2026-07-05,tier_s,knockout_stage,<NA>,BO3,-15,15,0,1,4,4
3,4,,2026-07-05,tier_s,knockout_stage,<NA>,BO3,0,0,0,1,4,4
4,5,,2026-07-04,tier_s,group_stage,<NA>,BO1,10,-10,0,2,5,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
520,521,,2017-01-20,tier_c,fase_2,<NA>,BO1,-4,4,33,205,104,104
521,522,,2017-01-20,tier_c,fase_2,<NA>,BO1,-4,4,33,204,153,153
522,523,,2017-01-20,tier_c,fase_2,<NA>,BO1,-4,4,33,208,151,151
523,524,,2017-01-20,tier_c,fase_1,<NA>,BO1,-4,4,33,209,200,200


In [684]:
df_pts.to_excel('../transform/excel_transform/partidas/Resultado_Partidas.xlsx', index=False)